# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch
import re
import tensorflow as tf
from models.llama3.generation import Llama
from tqdm import tqdm 
from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

/opt/anaconda3/envs/yt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

In [3]:
q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/balanced_dataset.csv')
queries = q_df['question']
label = q_df['label'].str.lower().map(label_mapper)

# Question Classifier

## Text-Classification

In [2]:
tokenizer = AutoTokenizer.from_pretrained("uw-vta/bloominzer-0.1")
model = AutoModelForSequenceClassification.from_pretrained("uw-vta/bloominzer-0.1")
bloominzer = pipeline("text-classification", model=model, tokenizer=tokenizer)

Device set to use mps:0


In [3]:
print(bloominzer("If I have 2 pair of apple, can i make apple pie with it?"))

[{'label': 'Synthesis', 'score': 0.9990537762641907}]


## LLM Classification

## Zero-Shot

### BART

In [4]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-mnli")
model = AutoModelForSequenceClassification.from_pretrained("facebook/bart-large-mnli")

bart = pipeline("zero-shot-classification",
                      model=model , tokenizer=tokenizer)

Device set to use mps:0


In [5]:
sequence_to_classify = "If I have 2 pair of apple, can i make apple pie with it?"
candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
label = bart(sequence_to_classify, candidate_labels)

In [6]:
label['labels'][0]

'application'

### mDeBERTa-v3-base-mnli-xnli

In [7]:
tokenizer = AutoTokenizer.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
model = AutoModelForSequenceClassification.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

ya_classifier = pipeline("zero-shot-classification",
                      model=model , tokenizer=tokenizer)

Device set to use mps:0


In [ ]:
sequence_to_classify = "If I have 2 pair of apple, can i make apple pie with it?"
candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
label = ya_classifier(sequence_to_classify, candidate_labels)

In [11]:
label['labels'][0]

'application'

## Text Generation

### LLAMA

In [2]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="modularai/Llama-3.1-8B-Instruct-GGUF",
	filename="llama-3.1-8b-instruct-q4_k_m.gguf"
)


ConnectionError: (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /api/models/modularai/Llama-3.1-8B-Instruct-GGUF (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x353fd7670>: Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: 102eac0f-e81e-4f2d-bcc4-b16224e0a925)')

In [ ]:
query

In [ ]:
query = 'If I have 2 pair of apple, can i make apple pie with it?'

llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": f"""
            Classify this query's Bloom's taxanomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            query : {query}

            Context: Bloom's taxonomy with examples:
            1. **Knowledge** (Recall facts): 
               - "Name three Shakespeare plays"
               - "What year did WWII end?"
               
            2. **Comprehension** (Understand meaning):
               - "Explain how photosynthesis works"
               - "Summarize the main theme of 1984"
               
            3. **Application** (Use concepts):
               - "Calculate the interest on a $5000 loan"
               - "How would you apply Newton's laws to this scenario?"
               
            4. **Analysis** (Examine structure):
               - "Compare Marxist vs capitalist economic models"
               - "Identify logical fallacies in this argument"
               
            5. **Synthesis** (Create new patterns):
               - "Design an experiment to test water purity"
               - "Propose a city planning solution for traffic"
               
            6. **Evaluation** (Make judgments):
               - "Which algorithm is most efficient for this task?"
               - "Assess the ethical implications of AI surveillance"

            Decision Framework:
            1. Identify the CORE ACTION:
               - Remember/List → Knowledge
               - Explain/Paraphrase → Comprehension
               - Apply/Use/Calculate → Application
               - Compare/Contrast → Analysis
               - Create/Design → Synthesis
               - Judge/Recommend → Evaluation

            2. Ask: Does the query require...
               - Simple recall? → Knowledge
               - Basic understanding? → Comprehension
               - Practical implementation? → Application
               - Breaking into components? → Analysis
               - Original construction? → Synthesis
               - Critical assessment? → Evaluation

            Respond ONLY with the exact taxonomy level name. No explanations.
            """
        }
    ]
)

llama_perf_context_print:        load time =    6889.15 ms
llama_perf_context_print: prompt eval time =    6888.89 ms /   433 tokens (   15.91 ms per token,    62.85 tokens per second)
llama_perf_context_print:        eval time =      58.27 ms /     1 runs   (   58.27 ms per token,    17.16 tokens per second)
llama_perf_context_print:       total time =    6948.07 ms /   434 tokens


{'id': 'chatcmpl-991d46d0-d2af-4e1e-a57d-77f9a71d5c87',
 'object': 'chat.completion',
 'created': 1748259754,
 'model': '/Users/pranitdas/.cache/huggingface/hub/models--modularai--Llama-3.1-8B-Instruct-GGUF/snapshots/966694508430d1177f6d585de779250e7a34bc3a/./llama-3.1-8b-instruct-q4_k_m.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant', 'content': 'Analysis'},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 433, 'completion_tokens': 1, 'total_tokens': 434}}

### DeepSeek

In [3]:
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")

r1_classifier = pipeline("text-generation", model=model , tokenizer=tokenizer)

KeyboardInterrupt: 

In [3]:
import re

query = 'What is the capital of india?'

# Step 1: Ask for classification
messages = [
    {
        "role": "user", 
        "content": f"""
Classify this query's Bloom's taxonomy level using ONLY one word from: 
[knowledge, comprehension, application, analysis, synthesis, evaluation]
Respond with only one word.
query: {query}""",
    }
]
data = r1_classifier(messages)

# Step 2: Extract assistant's response
assistant_content = next(
    item['content'] for item in data[0]['generated_text'] 
    if item['role'] == 'assistant'
).strip().lower()

# Step 3: Extract the classification using regex
bloom_keywords = ['knowledge', 'comprehension', 'application', 
                 'analysis', 'synthesis', 'evaluation']

classification = re.search(
    r'\b(' + '|'.join(bloom_keywords) + r')\b',
    assistant_content,
    flags=re.IGNORECASE
).group(1).lower()

print(classification)

knowledge


In [30]:
query = 'What is the capital of india?'

messages = [
    {
        "role": "user", 
        "content": f'''
        Classify this query's Bloom's taxanomy level using ONLY one word from: 
        [knowledge, comprehension, application, analysis, synthesis, evaluation]
        Respond ONLY with the exact taxonomy level name. No explanations
        query: {query}.''',
    }
]
data = r1_classifier(messages)

assistant_content = next(
    item['content'] for item in data[0]['generated_text'] if item['role'] == 'assistant'
)

message = [
    {
        "role": "user", 
        "content": f'''
        Extract one word bloom's taxanomy level from the response: 
        [knowledge, comprehension, application, analysis, synthesis, evaluation]
        Respond ONLY with the exact taxonomy level name. No explanations.
        response: {assistant_content}''',
    }
]

result = r1_classifier(message)
context = result[0]['generated_text'][-1]['content']
pattern = r"Answer[:\s]+(.+)"
match = re.search(pattern, context)
bloom_level = match.group(1).strip() if match else None
print(bloom_level)

None


In [29]:
assistant_content

'Alright, so I need to figure out the Bloom\'s Taxanomy level for the query "What is the capital of India?". Let me start by recalling what Bloom\'s Taxanomy is all about. It\'s a classification system for educational objectives, developed by Benjamin Bloom and his colleagues. The levels are: knowledge, comprehension, application, analysis, synthesis, and evaluation.\n\nThe query given is a straightforward question. It\'s asking for information about a specific place, the capital of India. So, I need to determine where this falls in the Taxanomy scale.\n\nFirst, I think about what "capital" means. In taxonomic terms, knowledge refers to recalling or knowing information, which is pretty basic. Comprehension is understanding what something means, which is a bit more advanced. Application is using knowledge in a specific context, so that\'s also in the same range as comprehension.\n\nNow, the question is not just asking what the capital is, but also what it is. So it\'s not just recalling

In [8]:
# Extract the assistant's final answer
result = r1_classifier(messages)  # Your second classifier call
bloom_level = result[0]['generated_text'][-1]['content']  # Returns "knowledge"

In [18]:
pattern = r"Answer[:\s]+(.+)"
match = re.search(pattern, bloom_level)
l = match.group(1).strip() if match else None
print(l)

** knowledge


### OWEN

In [7]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

qwen_classifier = pipeline("text-generation", model=model , tokenizer=tokenizer)

Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.32s/it]
Device set to use mps:0


In [ ]:
query = 'What is the capital of india?'

messages = [
    {
        "role": "user", 
        "content": f'''
        Classify this query's Bloom's taxanomy level using ONLY one word from: 
        [knowledge, comprehension, application, analysis, synthesis, evaluation]
        Respond ONLY with the exact taxonomy level name. No explanations
        query: {query}.''',
    }
]
message = qwen_classifier(messages)

print(message[0]['generated_text'][1]['content'])

### Predict Label Assignment

In [ ]:
# Reinforce 1
def q_classifier(query, prev_res):
    messages = [
        {
            "role": "user", 
            "content": f"""
            REVISE YOUR CLASSIFICATION. Your previous response '{prev_res}' was INVALID. 
            Classify this query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            CRITICAL RULES:
            1. MUST select from the 6 specified terms - NO exceptions
            2. Use these precise definitions:
            - knowledge: Recalling facts, terms, basic concepts (identify, list, name)
            - comprehension: Explaining meaning (describe, discuss, summarize)
            - application: Using information in new situations (execute, implement, solve)
            - analysis: Drawing connections among ideas (differentiate, organize, attribute)
            - synthesis: Producing new patterns (design, construct, integrate)
            - evaluation: Making judgments with evidence (appraise, defend, recommend)
            3. If multiple levels apply, choose the HIGHEST appropriate level
            4. Respond ONLY with the lowercase taxonomy word - NO other text

            Query: "{query}"

            Re-evaluate carefully. Your response MUST be exactly one word from the list.
            """
        }
    ]

    message = qwen_classifier(messages)
    
    return message[0]['generated_text'][1]['content']

In [ ]:
# Reinforce 2
def q1_classifier(query, prev_res):
    messages = [
        {
            "role": "user", 
            "content": f"""
            Predict Bloom's Taxonomy level of understanding. Classify query: {query} in one word. Responding {prev_res} is danger.
            Your entire response must be just one word chosen from following: 
            {label_mapper.keys()}
            {prev_res} is incorrect.
            """
        }
    ]

    message = qwen_classifier(messages)
    
    return message[0]['generated_text'][1]['content']

In [119]:
pred_labels= []
for query in tqdm(queries):
    messages = [
        {
            "role": "user", 
            "content": f'''
            Classify the query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            Consider these definitions:
            1. **knowledge**: Recalling facts/definitions (who, what, when, where)
            2. **comprehension**: Explaining concepts in own words (summarize, describe)
            3. **application**: Using knowledge in new situations (solve, compute, demonstrate)
            4. **analysis**: Breaking down concepts (compare, contrast, categorize)
            5. **synthesis**: Creating new patterns/solutions (design, develop, integrate)
            6. **evaluation**: Making judgments with criteria (justify, critique, recommend)

            Query: "{query}"

            Decision rules:
            - Focus on the query's PRIMARY cognitive demand
            - For multiple operations, choose the HIGHEST applicable level

            Respond ONLY with the exact taxonomy word in lowercase. No punctuation.
            ''',
        }
    ]

    message = qwen_classifier(messages)
    
    pred_labels.append(message[0]['generated_text'][1]['content'])

while(len(set(pred_labels)) != 6):
    for i , query in enumerate(queries):
        if(pred_labels[i].lower() not in label_mapper.keys()):
            prev_res = pred_labels[i]
            print(prev_res)
            pred_labels[i] = q_classifier(query, prev_res)
            if(pred_labels[i].lower() not in label_mapper.keys()):
                pred_labels[i] = q1_classifier(query, prev_res)
    print(set(pred_labels))

100%|██████████| 600/600 [08:16<00:00,  1.21it/s]


composition
explanation
comparison
definition
definition
definition
comparison
comparison
recognition
definition
definition
definition
{'comprehension', 'synthesis', 'comparison is evaluation.', 'knowledge', 'analysis', 'application', 'evaluation', 'definition is incorrect.'}
comparison is evaluation.
definition is incorrect.
comparison is evaluation.
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition'}
definition
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition is incorrect.'}
definition is incorrect.
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition'}
definition
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition'}
definition
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition is incorrect.'}
definition is incorrect.
{'comprehension', 'synthesis', 'knowledg

In [120]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       1.00      0.51      0.68       100
           1       0.72      0.43      0.54       100
           2       0.34      0.69      0.46       100
           3       0.60      0.79      0.68       100
           4       0.77      0.56      0.65       100
           5       0.86      0.72      0.78       100

    accuracy                           0.62       600
   macro avg       0.71      0.62      0.63       600
weighted avg       0.71      0.62      0.63       600



## Google-FLAN-T5-XL

In [4]:
model_name = "google/flan-t5-xl"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

flan_classifier = pipeline("text2text-generation", model=model , tokenizer=tokenizer, device=-1)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 14.42it/s]
Device set to use cpu


In [6]:
query = 'What is the capital of india?'

messages = f"""
    Classify query's Bloom's taxanomy level: 
    context: Bloom's taxanomy has 6 labels [knowledge, comprehension, application, analysis, synthesis, evaluation]
    query: {query}
"""

message = flan_classifier(messages)

### Predict Label Assignment

In [ ]:
pred_labels= []
for query in tqdm(queries):
    messages = f"""
            Classify query's Bloom's taxanomy level: 
            context: Bloom's taxanomy has 6 labels [knowledge, comprehension, application, analysis, synthesis, evaluation]
            query: {query}
        """

    message = flan_classifier(messages)
    
    pred_labels.append(message[0]['generated_text'].lower())

 10%|█         | 60/600 [08:35<1:09:42,  7.75s/it]